# The N-Ball Transformer — Predictions

**Pre-registration date:** 2026-07-29
**Rule:** All predictions entered here BEFORE the engine is run. Results —
confirmed or failed — remain in the record. Failed predictions stay in the
data, period, full stop.

Every prediction is decidable by running code. No conjectures, no open
problems.

---

## P1 — The step ratios are exactly π/2, symbolically

**Claim.** In exact symbolic arithmetic,

```
V(2)/V(1) = π/2        V(4)/V(2) = π/2
```

and the sequence breaks at the next step:

```
V(8)/V(4) = π²/12  ≠  π/2
```

**Falsification.** `sympy.simplify(V(a)/V(b) − π/2) ≠ 0` for either of the first
two, or `= 0` for the third.

**Test.** Symbolic simplification. No floating point is permitted in this test.

## P2 — The two-step recurrence is exact for every n

**Claim.** `V(n) = (2π/n) · V(n−2)` holds exactly, for every `n` tested, in
symbolic arithmetic.

This matters computationally: it gives `V` at all even `n` from `V(0) = 1` using
only multiplication, with no `Γ` evaluation at all.

**Falsification.** Any `n` at which the symbolic difference is non-zero.

**Test.** `n ∈ {2,4,6,8,10,12,16,24,32}` symbolically, plus a float
cross-check against the engine.

## P3 — The float implementation is correct to within a few ulp

**Claim.** The engine's `v_nball` differs from the correctly-rounded exact value
by a small number of units in the last place — and specifically, the ratios
`V(2)/V(1)` and `V(4)/V(2)` computed in float64 are **not bit-identical** to
`math.pi/2`, despite the identity being exact.

**Falsification.** Either an error exceeding 4 ulp at any tested `n`, or the
float ratios turning out to be bit-identical to `π/2` after all (which would
make the second half of the claim false).

**Test.** Compare `fp.v_nball(n)` against `float(V_exact(n))` for
`n = 0..16`, reporting the signed ulp difference.

## P4 — Root finding beats the engine's grid search

**Claim.** `n*` is the root of `ψ(n/2+1) − ln π`. The engine's grid search over
10,000 points is accurate only to about the grid spacing (`≈2×10⁻³`), giving an
`n*` in error by `~3×10⁻⁴`. A bracketed root solve (Brent) attains `~10⁻¹⁵`
using far fewer `digamma` evaluations.

**Falsification.** The grid result proving as accurate as the root solve, or the
root solve requiring more function evaluations than the grid.

**Test.** Run both, compare against each other and against the stored `N_STAR`,
and count `digamma` calls.

---

## Registered as executable assertions

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta import fixed_point as fp

import math
import numpy as np
import sympy as sy
import scipy.special as ss
import scipy.optimize as so

print('engine :', 'ValaQuenta/fixed_point.py')
print('python :', sys.version.split()[0])

In [ ]:
# Exact V(n) in rational/symbolic arithmetic -- no floating point at all.
def V_exact(k):
    """V(k) = pi^(k/2) / Gamma(k/2 + 1) as an exact sympy expression."""
    return sy.pi**sy.Rational(k, 2) / sy.gamma(sy.Rational(k, 2) + 1)

def V_float(x):
    """The engine's floating-point implementation."""
    return fp.v_nball(x)

In [ ]:
LN_PI = math.log(math.pi)

def P1_exact_ratios():
    a = sy.simplify(V_exact(2)/V_exact(1) - sy.pi/2) == 0
    b = sy.simplify(V_exact(4)/V_exact(2) - sy.pi/2) == 0
    c = sy.simplify(V_exact(8)/V_exact(4) - sy.pi/2) != 0
    d = sy.simplify(V_exact(8)/V_exact(4) - sy.pi**2/12) == 0
    return bool(a and b and c and d)


def P2_recurrence():
    for k in (2, 4, 6, 8, 10, 12, 16, 24, 32):
        if sy.simplify(V_exact(k) - sy.Rational(2, k)*sy.pi*V_exact(k-2)) != 0:
            return False
    return True


def _ulp_diff(got, exact):
    """Signed difference in units in the last place."""
    if got == exact:
        return 0.0
    return (got - exact) / math.ulp(exact)


def P3_float_within_ulp(max_ulp=4.0):
    for k in range(0, 17):
        exact = float(V_exact(k))
        got = V_float(k)
        if abs(_ulp_diff(got, exact)) > max_ulp:
            return False
    # second half: the float ratios are NOT bit-identical to pi/2
    r21 = V_float(2)/V_float(1)
    r42 = V_float(4)/V_float(2)
    return not (r21 == math.pi/2 and r42 == math.pi/2)


def P4_rootfind_beats_grid():
    f = lambda x: ss.digamma(x/2 + 1) - LN_PI
    root = so.brentq(f, 1.0, 20.0, xtol=1e-15, rtol=8.9e-16)
    grid_n = fp.v_nball_peak()['n_star']
    err_grid = abs(grid_n - root)
    err_stored = abs(fp.N_STAR - root)
    return bool(err_grid > 1e-5 and err_stored < 1e-8)


PREDICTIONS = [
    ('P1', 'V(2)/V(1) = V(4)/V(2) = pi/2 exactly; breaks at pi^2/12', P1_exact_ratios),
    ('P2', 'V(n) = (2pi/n)V(n-2) exactly for every n',               P2_recurrence),
    ('P3', 'float impl within a few ulp; ratios not bit-identical',  P3_float_within_ulp),
    ('P4', 'root find far more accurate than the 10k-point grid',    P4_rootfind_beats_grid),
]

print(f'{len(PREDICTIONS)} predictions registered, none evaluated yet:')
for tag, desc, _ in PREDICTIONS:
    print(f'  {tag}  {desc}')